# 01 — Fine-tune HerBERT and extract out-of-sample predictions
Fine-tunes a transformer on the **noisy** labels and produces **out-of-sample** predicted distributions $p_i$ for every item via k-fold cross-fitting (each item is predicted by a model that never trained on it — avoids memorization leakage, mirroring confident learning's cross-validated probabilities).

Default model: `allegro/herbert-base-cased` (Polish, per the research plan). Switch to `FacebookAI/xlm-roberta-base` to match the official AlleNoise baseline.

**Output:** `../data/pi_memmap.npy` (float16, N×K) and `../data/item_meta.parquet`. The dense probability matrix is ~N×K×2 bytes (≈5–6 GB for the full set); set `SUBSET` while prototyping.

In [1]:
# Notebook: 01_finetune_and_predict
# Shared plotting style: grayscale seaborn, dpi 600, PNG + PDF, no captions.
import os, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"] = "0.2"; plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["font.family"] = "DejaVu Sans"
GREYS = ["#111111", "#555555", "#888888", "#bbbbbb", "#dddddd"]
FIG = os.path.join("..", "results", "figures"); TAB = os.path.join("..", "results", "tables")
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, f"{name}.{ext}"), dpi=600, bbox_inches="tight")
    plt.close(fig)

# --- config ---
import torch
DATA_DIR   = os.path.join("..", "data")
MODEL_NAME = "allegro/herbert-base-cased"   # or "FacebookAI/xlm-roberta-base"
MAX_LEN    = 64
N_FOLDS    = 3
EPOCHS     = 10          # loss가 안 꺾였으니 넉넉히; val이 평탄해지면 그게 수렴
LR         = 4e-5        # batch를 키웠으니 소폭 상향
BATCH      = 128         # OOM 나면 96 -> 64로 낮추세요
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SUBSET     = None
SEED       = 42
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| model:", MODEL_NAME)
assert DEVICE == "cuda", "GPU 필요"

# --- load normalized data & encode labels on the NOISY label space ---
import numpy as np
df = pd.read_parquet(os.path.join(DATA_DIR, "allenoise_norm.parquet"))
if SUBSET:
    df = df.sample(SUBSET, random_state=SEED).reset_index(drop=True)
classes = np.sort(df["noisy_category"].unique())
cls2idx = {c_: i for i, c_ in enumerate(classes)}
K = len(classes)
df["noisy_id"] = df["noisy_category"].map(cls2idx).astype(int)
df["clean_id"] = df["clean_category"].map(lambda c_: cls2idx.get(c_, -1)).astype(int)  # -1 if clean cat unseen
N = len(df)
print(f"N={N:,}  K={K:,}  (dense pi matrix ~ {N*K*2/1e9:.2f} GB float16)")

# --- tokenizer / dataset ---
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class TitleDS(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts); self.labels = labels
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = tok(self.texts[i], truncation=True, max_length=MAX_LEN,
                  padding="max_length", return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[i]))
        return item

# --- k-fold cross-fitting: train on noisy labels, predict held-out fold ---
from sklearn.model_selection import KFold
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
import gc, time

pi_path = os.path.join(DATA_DIR, "pi_memmap.npy")
try:
    del pi
except NameError:
    pass
gc.collect()
try:
    if os.path.exists(pi_path):
        os.remove(pi_path)
    pi = np.lib.format.open_memmap(pi_path, mode="w+", dtype=np.float16, shape=(N, K))
except PermissionError:
    pi_path = os.path.join(DATA_DIR, f"pi_memmap_{int(time.time())}.npy")
    pi = np.lib.format.open_memmap(pi_path, mode="w+", dtype=np.float16, shape=(N, K))
    print("기존 파일 잠김 -> 새 파일:", pi_path)

def predict_indices(model, idx):
    """held-out subset 예측 확률 (모니터링/저장 공용)."""
    model.eval()
    dl = DataLoader(TitleDS(df["text"].values[idx]), batch_size=BATCH,
                    shuffle=False, num_workers=0, pin_memory=True)
    outs = []
    with torch.no_grad():
        for batch in dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.cuda.amp.autocast():
                logits = model(**batch).logits
            outs.append(torch.softmax(logits.float(), 1).cpu().numpy().astype(np.float16))
    return np.concatenate(outs, 0)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (tr, te) in enumerate(kf.split(np.arange(N))):
    print(f"\n===== fold {fold+1}/{N_FOLDS} =====")
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=K).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    dl_tr = DataLoader(TitleDS(df["text"].values[tr], df["noisy_id"].values[tr]),
                       batch_size=BATCH, shuffle=True, num_workers=0, pin_memory=True)
    total = EPOCHS * len(dl_tr)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * total), total)
    scaler = torch.cuda.amp.GradScaler()

    # 수렴 모니터링용 고정 val 서브셋 (held-out fold에서 4000개)
    rng_v = np.random.default_rng(SEED)
    vsel = te[rng_v.choice(len(te), size=min(4000, len(te)), replace=False)]

    for ep in range(EPOCHS):
        model.train(); run_loss = 0.0
        for batch in tqdm(dl_tr, desc=f"fold{fold+1} ep{ep+1}"):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            opt.zero_grad()
            with torch.cuda.amp.autocast():
                loss = model(**batch).loss
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            run_loss = 0.98 * run_loss + 0.02 * loss.item()
        vp = predict_indices(model, vsel)
        v_clean = (vp.argmax(1) == df["clean_id"].values[vsel]).mean()
        print(f"  ep{ep+1}: train_loss~{run_loss:.3f}  val_clean_acc={v_clean:.3f}")

    # held-out fold 전체 예측 저장
    probs_te = predict_indices(model, te)
    pi[te] = probs_te
    acc_noisy = (probs_te.argmax(1) == df["noisy_id"].values[te]).mean()
    acc_clean = (probs_te.argmax(1) == df["clean_id"].values[te]).mean()
    print(f"  [fold done] held-out top-1 vs noisy={acc_noisy:.3f} vs clean={acc_clean:.3f}")
    del model; torch.cuda.empty_cache()
pi.flush()
print("\nsaved ->", pi_path)

# --- per-item metadata for downstream notebooks ---
rows = np.arange(N)
p_noisy = pi[rows, df["noisy_id"].values].astype(np.float32)
ent = -np.sum(np.clip(pi.astype(np.float32), 1e-12, 1) *
              np.log(np.clip(pi.astype(np.float32), 1e-12, 1)), axis=1)
pred_arg = np.asarray(pi).argmax(axis=1)
meta = pd.DataFrame({
    "idx": rows,
    "noisy_id": df["noisy_id"].values,
    "clean_id": df["clean_id"].values,
    "is_misregistered": (df["noisy_id"].values != df["clean_id"].values).astype(int),
    "p_noisy": p_noisy,
    "entropy": ent.astype(np.float32),
    "pred_argmax": pred_arg,
})
meta.to_parquet(os.path.join(DATA_DIR, "item_meta.parquet"))
np.save(os.path.join(DATA_DIR, "label_classes.npy"), classes)
print("saved item_meta.parquet and label_classes.npy")
print(meta.head())

device: cuda | model: allegro/herbert-base-cased
N=502,310  K=5,691  (dense pi matrix ~ 5.72 GB float16)


C:\Users\miy\miniconda3\envs\seller_seg\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



===== fold 1/3 =====


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33169.07it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arc

  ep1: train_loss~4.813  val_clean_acc=0.270


...

  ep8: train_loss~1.562  val_clean_acc=0.565


fold3 ep9: 100%|██████████| 2617/2617 [15:15<00:00,  2.86it/s]


  ep9: train_loss~1.491  val_clean_acc=0.570


fold3 ep10: 100%|██████████| 2617/2617 [15:17<00:00,  2.85it/s]


  ep10: train_loss~1.449  val_clean_acc=0.570
  [fold done] held-out top-1 vs noisy=0.631 vs clean=0.553

saved -> ..\data\pi_memmap.npy
saved item_meta.parquet and label_classes.npy
   idx  noisy_id  clean_id  is_misregistered   p_noisy   entropy  pred_argmax
0    0         0         0                 0  0.002176  5.846993           39
1    1         0         0                 0  0.003332  5.799827         2249
2    2         0         0                 0  0.000485  6.389935          278
3    3         0         0                 0  0.000937  5.688538         3832
4    4         0         0                 0  0.000628  4.909289          264
